In [1]:
# =========================================================
# ADVANCED MULTI-LABEL TOXIC COMMENT CLASSIFICATION
# BEST MODEL SELECTION + ADVANCED VISUALIZATION
# =========================================================

# =========================
# IMPORT LIBRARIES
# =========================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings

warnings.filterwarnings("ignore")

from datasets import load_dataset

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.multiclass import OneVsRestClassifier

from sklearn.pipeline import Pipeline

# =========================
# MACHINE LEARNING MODELS
# =========================

from sklearn.linear_model import LogisticRegression

from sklearn.naive_bayes import MultinomialNB

from sklearn.svm import LinearSVC

from sklearn.ensemble import RandomForestClassifier

# =========================
# EVALUATION METRICS
# =========================

from sklearn.metrics import accuracy_score

from sklearn.metrics import hamming_loss

from sklearn.metrics import classification_report

from sklearn.metrics import multilabel_confusion_matrix

from sklearn.metrics import f1_score

# =========================
# CREATE FOLDERS
# =========================

os.makedirs("models", exist_ok=True)

os.makedirs("visualizations", exist_ok=True)

# =========================
# LOAD DATASET
# =========================

print("=" * 60)
print("LOADING DATASET...")
print("=" * 60)

dataset = load_dataset(
    "thesofakillers/jigsaw-toxic-comment-classification-challenge"
)

train_data = dataset["train"]

df = pd.DataFrame(train_data)

print("\nDATASET LOADED SUCCESSFULLY!")

print("\nTOTAL ROWS:", len(df))

# =========================
# OPTIONAL: SMALL SAMPLE
# Faster Training
# =========================

df = df.sample(
    30000,
    random_state=42
)

# =========================
# LABELS
# =========================

labels = [

    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"

]

# =========================
# TEXT LENGTH FEATURE
# =========================

df["text_length"] = df["comment_text"].apply(len)

# =========================================================
# ADVANCED DATA VISUALIZATION
# =========================================================

# =========================
# 1. LABEL DISTRIBUTION
# =========================

label_counts = df[labels].sum()

plt.figure(figsize=(10, 6))

sns.barplot(
    x=label_counts.index,
    y=label_counts.values
)

plt.title("Toxic Label Distribution")

plt.xticks(rotation=45)

plt.tight_layout()

plt.savefig(
    "visualizations/label_distribution.png"
)

plt.close()

# =========================
# 2. CORRELATION HEATMAP
# =========================

plt.figure(figsize=(10, 6))

sns.heatmap(

    df[labels].corr(),

    annot=True,

    cmap="coolwarm"

)

plt.title("Correlation Heatmap")

plt.tight_layout()

plt.savefig(
    "visualizations/heatmap.png"
)

plt.close()

# =========================
# 3. TEXT LENGTH DISTRIBUTION
# =========================

plt.figure(figsize=(10, 6))

sns.histplot(

    df["text_length"],

    bins=50,

    kde=True

)

plt.title("Comment Length Distribution")

plt.xlabel("Text Length")

plt.ylabel("Frequency")

plt.tight_layout()

plt.savefig(
    "visualizations/text_length_distribution.png"
)

plt.close()

# =========================================================
# FEATURES & TARGETS
# =========================================================

X = df["comment_text"]

y = df[labels]

# =========================
# TRAIN TEST SPLIT
# =========================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42

)

# =========================================================
# MACHINE LEARNING MODELS
# =========================================================

models = {

    "Logistic Regression":

        OneVsRestClassifier(

            LogisticRegression(
                max_iter=1000
            )

        ),

    "Naive Bayes":

        OneVsRestClassifier(

            MultinomialNB()

        ),

    "Linear SVM":

        OneVsRestClassifier(

            LinearSVC()

        ),

    "Random Forest":

        OneVsRestClassifier(

            RandomForestClassifier(
                n_estimators=50,
                random_state=42
            )

        )

}

# =========================================================
# MODEL TRAINING & COMPARISON
# =========================================================

best_accuracy = 0

best_model = None

best_model_name = ""

results = []

for name, classifier in models.items():

    print("\n" + "=" * 60)

    print(f"TRAINING: {name}")

    print("=" * 60)

    pipeline = Pipeline([

        (

            "tfidf",

            TfidfVectorizer(

                stop_words="english",

                max_features=10000

            )

        ),

        (

            "classifier",

            classifier

        )

    ])

    # =========================
    # TRAIN MODEL
    # =========================

    pipeline.fit(
        X_train,
        y_train
    )

    # =========================
    # PREDICTIONS
    # =========================

    predictions = pipeline.predict(
        X_test
    )

    # =========================
    # EVALUATION
    # =========================

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    h_loss = hamming_loss(
        y_test,
        predictions
    )

    f1 = f1_score(
        y_test,
        predictions,
        average="micro"
    )

    print(f"\nAccuracy: {round(accuracy * 100, 2)} %")

    print(f"Hamming Loss: {round(h_loss, 4)}")

    print(f"F1 Score: {round(f1, 4)}")

    results.append({

        "Model": name,
        "Accuracy": accuracy,
        "Hamming Loss": h_loss,
        "F1 Score": f1

    })

    # =========================
    # SAVE BEST MODEL
    # =========================

    if accuracy > best_accuracy:

        best_accuracy = accuracy

        best_model = pipeline

        best_model_name = name

# =========================================================
# DISPLAY BEST MODEL
# =========================================================

print("\n" + "=" * 60)

print("BEST MODEL")

print("=" * 60)

print(f"\nBest Model: {best_model_name}")

print(f"Best Accuracy: {round(best_accuracy * 100, 2)} %")

# =========================================================
# SAVE BEST MODEL
# =========================================================

joblib.dump(

    best_model,

    "models/best_toxic_model.pkl"

)

print("\nBEST MODEL SAVED SUCCESSFULLY!")

# =========================================================
# RESULTS DATAFRAME
# =========================================================

results_df = pd.DataFrame(results)

print("\nMODEL COMPARISON:\n")

print(results_df)

# =========================================================
# MODEL COMPARISON GRAPH
# =========================================================

plt.figure(figsize=(10, 6))

sns.barplot(

    data=results_df,

    x="Model",

    y="Accuracy"

)

plt.title("Model Accuracy Comparison")

plt.xticks(rotation=15)

plt.tight_layout()

plt.savefig(
    "visualizations/model_comparison.png"
)

plt.close()

# =========================================================
# CONFUSION MATRICES
# =========================================================

best_predictions = best_model.predict(
    X_test
)

matrices = multilabel_confusion_matrix(
    y_test,
    best_predictions
)

for i, matrix in enumerate(matrices):

    plt.figure(figsize=(5, 4))

    sns.heatmap(

        matrix,

        annot=True,

        fmt="d",

        cmap="Blues"

    )

    plt.title(
        f"Confusion Matrix - {labels[i]}"
    )

    plt.tight_layout()

    plt.savefig(
        f"visualizations/confusion_matrix_{labels[i]}.png"
    )

    plt.close()

# =========================================================
# CLASSIFICATION REPORT
# =========================================================

print("\nCLASSIFICATION REPORT:\n")

print(

    classification_report(

        y_test,
        best_predictions,

        target_names=labels

    )

)

print("\nPROJECT COMPLETED SUCCESSFULLY!")

LOADING DATASET...


Generating test split: 100%|██████████| 306328/306328 [00:03<00:00, 101762.60 examples/s]



DATASET LOADED SUCCESSFULLY!

TOTAL ROWS: 159571

TRAINING: Logistic Regression

Accuracy: 91.68 %
Hamming Loss: 0.0224
F1 Score: 0.5397

TRAINING: Naive Bayes

Accuracy: 90.8 %
Hamming Loss: 0.0266
F1 Score: 0.388

TRAINING: Linear SVM

Accuracy: 92.03 %
Hamming Loss: 0.0193
F1 Score: 0.6631

TRAINING: Random Forest

Accuracy: 91.78 %
Hamming Loss: 0.0191
F1 Score: 0.6631

BEST MODEL

Best Model: Linear SVM
Best Accuracy: 92.03 %

BEST MODEL SAVED SUCCESSFULLY!

MODEL COMPARISON:

                 Model  Accuracy  Hamming Loss  F1 Score
0  Logistic Regression  0.916833      0.022361  0.539737
1          Naive Bayes  0.908000      0.026556  0.387964
2           Linear SVM  0.920333      0.019278  0.663107
3        Random Forest  0.917833      0.019111  0.663075

CLASSIFICATION REPORT:

               precision    recall  f1-score   support

        toxic       0.87      0.60      0.71       543
 severe_toxic       0.52      0.26      0.35        53
      obscene       0.90      0.63  